# Growth Prediction Model — OD680 Forecast

**Goal:** predict OD680 in 60 minutes given the current sensor snapshot.

**Output contract** (consumed by `run_ml_models` in `agent/nodes.py`):
```python
ml_outputs['growth_prediction'] = {
    'OD680': {
        'current':  float,
        'in_60min': float,
        'trend':    'Rising' | 'Stable' | 'Declining',
    }
}
```

See `docs/phase_b_ml_interface.md` for the full interface spec.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Make project root importable
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Generate (or load) training data

Replace the simulator with a real CSV once container logging is available:
```python
df = pd.read_csv('../data/raw/container_logs.csv', parse_dates=['timestamp'])
```

In [ ]:
# ----- Physics-based simulator (replace with real data when available) -----
# Monod-type batch model for Spirulina platensis
# Reference parameters: Cornet (1992), Richmond (2004)

SEED       = 42
N_CYCLES   = 60          # number of simulated batch cycles
DT_MIN     = 10          # sampling interval (minutes)
CYCLE_DAYS = 5           # duration of one batch

rng = np.random.default_rng(SEED)


def simulate_batch(rng, dt_min=10, days=5):
    """Simulate one batch cycle. Returns a DataFrame of sensor readings."""
    steps   = int(days * 24 * 60 / dt_min)
    t_hours = np.arange(steps) * dt_min / 60

    # Light follows a sinusoidal day/night cycle
    light_lux = np.clip(
        12000 * np.sin(np.pi * ((t_hours % 24) / 12)) + rng.normal(0, 400, steps),
        0, None,
    )

    # Temperature: warmer during the day
    temperature_c = (
        33.0
        + 3.0 * np.sin(np.pi * ((t_hours % 24) / 12))
        + rng.normal(0, 0.3, steps)
    )

    # Growth rate: Monod-type (light and temp limited)
    mu_max     = 0.06   # h^-1  (realistic for Sp. in photobioreactor)
    K_light    = 3000   # lux   (half-saturation constant)
    T_opt      = 35.0   # °C
    T_range    = 10.0   # °C (half-width of thermal window)

    mu = (
        mu_max
        * (light_lux / (K_light + light_lux))
        * np.exp(-((temperature_c - T_opt) / T_range) ** 2)
    )

    # Integrate OD680 using Euler method
    od680 = np.zeros(steps)
    od680[0] = rng.uniform(0.05, 0.15)   # random inoculum density
    dt_h = dt_min / 60
    for i in range(1, steps):
        od680[i] = od680[i-1] + mu[i-1] * od680[i-1] * dt_h
    od680 += rng.normal(0, 0.005, steps)   # sensor noise
    od680 = np.clip(od680, 0.01, 2.0)

    # pH: rises with photosynthesis during the day
    ph_base = 9.0 + 0.3 * np.sin(np.pi * ((t_hours % 24) / 12))
    ph_base += 0.002 * t_hours   # slight drift up as CO2 is consumed
    ph = np.clip(ph_base + rng.normal(0, 0.05, steps), 7.0, 11.5)

    # Conductivity: slight rise as water evaporates
    conductivity_ms = 20.0 + 0.01 * t_hours + rng.normal(0, 0.2, steps)

    # Dissolved O2: correlated with photosynthesis
    dissolved_o2 = np.clip(
        85.0 + 15.0 * np.sin(np.pi * ((t_hours % 24) / 12)) + rng.normal(0, 2, steps),
        20, 150,
    )

    return pd.DataFrame({
        't_hours':         t_hours,
        'od680':           od680,
        'ph':              ph,
        'temperature_c':   temperature_c,
        'conductivity_ms': conductivity_ms,
        'dissolved_o2':    dissolved_o2,
        'light_lux':       light_lux,
    })


cycles = [simulate_batch(rng) for _ in range(N_CYCLES)]
df_raw = pd.concat(cycles, ignore_index=True)
print(f"Generated {len(df_raw):,} rows from {N_CYCLES} batch cycles")
df_raw.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))
cols  = ['od680', 'ph', 'temperature_c', 'conductivity_ms', 'dissolved_o2', 'light_lux']
for ax, col in zip(axes.flat, cols):
    ax.hist(df_raw[col], bins=60, edgecolor='none', alpha=0.8)
    ax.set_xlabel(col)
    ax.set_ylabel('count')
plt.suptitle('Simulated sensor distributions', fontsize=13)
plt.tight_layout()
plt.show()

# Sample one cycle
one = cycles[0]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(one['t_hours'], one['od680'], linewidth=1.5)
ax.set_xlabel('Hours into batch')
ax.set_ylabel('OD680')
ax.set_title('Example batch growth curve')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

Target: `od680` shifted back by 6 steps (= 60 min at 10-min intervals).

In [ ]:
HORIZON_STEPS = 6   # 6 × 10 min = 60 min forecast horizon

FEATURE_COLS = ['od680', 'ph', 'temperature_c', 'conductivity_ms', 'dissolved_o2', 'light_lux']


def build_features(df, horizon=HORIZON_STEPS, feature_cols=FEATURE_COLS):
    """Create X (current snapshot) and y (OD680 in `horizon` steps)."""
    X = df[feature_cols].copy()

    # Rate-of-change features (1-step delta)
    for col in ('od680', 'ph', 'temperature_c'):
        X[f'd_{col}'] = df[col].diff().fillna(0)

    # Target: OD680 `horizon` steps ahead
    y = df['od680'].shift(-horizon)

    # Drop rows where target is NaN (end of cycle)
    mask = y.notna()
    return X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)


X, y = build_features(df_raw)
print(f"Features shape: {X.shape}")
print(f"Target range:   {y.min():.3f} – {y.max():.3f}")
X.head()

## 4. Train / Validation Split

Temporal split — last 20 % of cycles as validation set.

In [ ]:
from sklearn.model_selection import train_test_split

# Temporal split (not random — avoid data leakage from autocorrelation)
split = int(len(X) * 0.80)
X_train, X_val = X.iloc[:split], X.iloc[split:]
y_train, y_val = y.iloc[:split], y.iloc[split:]

print(f"Train: {len(X_train):,}  |  Val: {len(X_val):,}")

## 5. Model Training

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics  import mean_absolute_error, r2_score

model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=SEED,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
mae  = mean_absolute_error(y_val, y_pred)
r2   = r2_score(y_val, y_pred)
print(f"Validation MAE : {mae:.4f} OD units")
print(f"Validation R²  : {r2:.4f}")

## 6. Evaluation

In [ ]:
# Parity plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.scatter(y_val, y_pred, alpha=0.15, s=5)
lim = [y_val.min(), y_val.max()]
ax.plot(lim, lim, 'r--', linewidth=1)
ax.set_xlabel('Actual OD680 (60 min)')
ax.set_ylabel('Predicted OD680')
ax.set_title(f'Parity plot  (R²={r2:.3f}, MAE={mae:.4f})')

# Residuals
ax = axes[1]
residuals = y_pred - y_val
ax.hist(residuals, bins=60, edgecolor='none')
ax.axvline(0, color='red', linewidth=1)
ax.set_xlabel('Residual (pred – actual)')
ax.set_ylabel('Count')
ax.set_title('Residual distribution')

plt.tight_layout()
plt.show()

# Feature importance
fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)
fi.plot.barh(figsize=(8, 5))
plt.xlabel('Feature importance')
plt.title('Gradient Boosting — feature importances')
plt.tight_layout()
plt.show()

## 7. Save model + wire into agent interface

In [ ]:
import joblib
from pathlib import Path

MODEL_PATH = ROOT / 'ml' / 'growth' / 'model.pkl'
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({'model': model, 'feature_cols': list(X.columns)}, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

In [ ]:
# ---- Verify the agent contract ----
# This is the exact function that ml/growth/__init__.py will expose.

TREND_THRESHOLD = 0.02   # OD units — below this = Stable


def predict_growth(sensor: dict) -> dict:
    """Predict OD680 in 60 min. Returns ml_outputs['growth_prediction']."""
    artifact = joblib.load(MODEL_PATH)
    mdl      = artifact['model']
    fcols    = artifact['feature_cols']

    row = pd.DataFrame([{
        'od680':           sensor.get('od680',           0.8),
        'ph':              sensor.get('ph',               9.5),
        'temperature_c':   sensor.get('temperature_c',   33.0),
        'conductivity_ms': sensor.get('conductivity_ms', 20.0),
        'dissolved_o2':    sensor.get('dissolved_o2_pct', 90.0),
        'light_lux':       sensor.get('light_lux',       10000),
        'd_od680':         0.0,
        'd_ph':            0.0,
        'd_temperature_c': 0.0,
    }])[fcols]   # reorder to match training columns

    pred = float(mdl.predict(row)[0])
    delta = pred - sensor.get('od680', 0.8)

    if delta > TREND_THRESHOLD:
        trend = 'Rising'
    elif delta < -TREND_THRESHOLD:
        trend = 'Declining'
    else:
        trend = 'Stable'

    return {
        'OD680': {
            'current':  round(sensor.get('od680', 0.8), 3),
            'in_60min': round(pred, 3),
            'trend':    trend,
        }
    }


# Smoke test against the test-healthy container reading
test_sensor = {
    'ph': 9.5, 'temperature_c': 33.0, 'od680': 0.85,
    'conductivity_ms': 20.0, 'dissolved_o2_pct': 95.0, 'light_lux': 10000,
}
result = predict_growth(test_sensor)
print("Contract output:")
import json; print(json.dumps(result, indent=2))

assert 'OD680' in result
assert result['OD680']['trend'] in ('Rising', 'Stable', 'Declining')
print("Contract verification PASSED")

## 8. Next steps

- [ ] Retrain on real container data when 30+ cycles are available
- [ ] Replace `GradientBoostingRegressor` with LightGBM for speed
- [ ] Use a rolling window (last 6 readings) instead of a single snapshot
- [ ] Cross-validate with `TimeSeriesSplit` from sklearn
- [ ] Copy `predict_growth` body into `ml/growth/__init__.py`
- [ ] Wire into `agent/nodes.py` `run_ml_models` following `docs/phase_b_ml_interface.md`